# Double-Pendulum Dynamics — Training (Colab)

Trains the NeRD-style dynamics model for either modality via one config switch.
The structure mirrors the **Neural Robot Dynamics** repo
([NVlabs/neural-robot-dynamics](https://github.com/NVlabs/neural-robot-dynamics)):

| NeRD | here |
|------|------|
| `train/train.py` (build cfg → trainer → `algo.train()`) | final cell |
| `VanillaTrainer` / `SequenceModelTrainer` | `Trainer` |
| `ModelMixedInput` | `DynamicsModel` (`models/dynamics.py`) |
| `TrajectoryDataset` (sequence sampling) | `TrajectoryDataset` |
| `RunningMeanStd` + `compute_dataset_statistics()` | same |
| `compute_loss` (MSE + variance weighting + itemized) | same |
| `one_epoch(train=...)` / `get_scheduled_learning_rate` | same |

**Both modalities share one pipeline** — identical trajectory windows and
train/val split — so `state` vs `vision` is a clean, matched comparison.

- **Target Δs (18d):** `[R_j0(6), R_j1(6), w0(3), w1(3)]`, additive delta. Base
  pose dropped (fixed base); contacts / torque / gravity are inputs only.
- **Dense supervision (NeRD):** the causal GPT predicts a Δs at every position
  in the length-`sample_sequence_length` window.
- **Logging:** Weights & Biases (`wandb`) + a local `metrics.jsonl` mirror under
  `runs/<name>/`. wandb is optional — set `use_wandb=False` (or skip login) and it
  falls back to jsonl only.

> Multi-step autoregressive rollout eval (NeRD's `NeuralSimEvaluator`) lives in
> `eval.py` at the repo root — it needs the MuJoCo env, so run it locally on the
> saved checkpoints (`python eval.py --ckpt runs/<name>/best_model.pt`).
> `evaluate()` here is single-step validation on a fixed random subset of the
> held-out windows.

## 1. Colab setup

Get the code + data onto the machine, then install the two deps Colab lacks
(`torch`/`numpy`/`tqdm` are preinstalled). Training does **not** import the env,
so MuJoCo/Gymnasium are unnecessary here.

Pick whichever data source applies and skip the rest.

In [ ]:
# --- Colab only: fetch code + data. Skip these lines when running locally. ---
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    # 1) Code — clone the repo (or `git pull` if already present).
    import os
    if not os.path.exists("double-pendulum"):
        !git clone https://github.com/veroses/double-pendulum.git
    %cd double-pendulum

    # 2) Deps not already in Colab.
    !pip -q install h5py hdf5plugin wandb

    # 3) Data — the 30 shards are ~6 GB, too big for git. Mount Drive (recommended)
    #    and point DATA_DIR at the folder holding shard_000.h5 ... shard_029.h5.
    from google.colab import drive
    drive.mount("/content/drive")
    # e.g. DATA_DIR = "/content/drive/MyDrive/double_pendulum/data"


In [ ]:
import os, sys
# Repo root must be importable so `models/` and its siblings resolve.
REPO_ROOT = os.getcwd()
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "models"))  # models import each other flat
print("repo root:", REPO_ROOT)


## 2. Imports

In [ ]:
import glob, json, math, time
from dataclasses import dataclass, field, asdict

import h5py
import hdf5plugin  # noqa: F401  registers the LZ4 filter so rgb_* datasets decode
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    import wandb
except Exception:
    wandb = None  # notebook still runs; logging falls back to metrics.jsonl

from dynamics import (
    FRAMES_KEY, STATE_KEY, TORQUE_KEY,
    DynamicsModel, DynamicsModelConfig,
)
from encoders import MLPEncoderConfig, VisionEncoderConfig
from transformer import GPTConfig


## 3. Config

One dataclass standing in for NeRD's `cfg/<Env>/transformer.yaml`. Edit here or
override the instance in the run cell.

In [ ]:
@dataclass
class Config:
    # --- data ---
    modality: str = "state"          # "state" | "vision"
    data_dir: str = "data"           # folder with shard_000.h5 ... shard_029.h5
    val_frac: float = 0.05
    seed: int = 0

    # --- sequence (NeRD: sample_sequence_length) ---
    sample_sequence_length: int = 10  # transitions per window == GPT context

    # --- model: causal GPT dynamics core (NeRD Table 2, pendulum column) ---
    n_layer: int = 6
    n_head: int = 12
    n_embd: int = 192
    block_size: int = 32
    dropout: float = 0.0
    head_hidden: tuple = (64,)

    # --- optimisation ---
    batch_size: int = 0               # 0 -> 256 (state) / 64 (vision)
    lr_start: float = 1e-3
    lr_end: float = 1e-4
    lr_schedule: str = "linear"       # "constant" | "linear" | "cosine"
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.95)
    num_epochs: int = 100
    num_batches_per_epoch: int = 500  # NeRD draws a fixed #batches/epoch
    num_val_batches: int = 50
    truncate_grad: bool = True
    grad_norm: float = 1.0
    # Mixed precision: fp16 autocast + GradScaler on CUDA (T4 tensor cores give
    # ~2x vision throughput). Training only — validation always runs fp32 so the
    # reported metrics and best-checkpoint selection are exact. Ignored (clean
    # no-op) on MPS/CPU.
    use_amp: bool = True
    # NeRD robustness trick: perturb the (normalised) inputs with small Gaussian
    # noise during training so autoregressive rollout — where the model sees its
    # own slightly-off predictions — stays on-distribution. Training only;
    # validation always runs clean.
    inject_noise: bool = True

    # --- normalisation ---
    compute_dataset_statistics: bool = True
    normalize_input: bool = True
    normalize_output: bool = True
    stat_trajs: int = 1000            # trajectories sampled to fit RMS stats

    # --- logging (Weights & Biases) / checkpoints ---
    use_wandb: bool = True            # False -> metrics.jsonl only
    wandb_project: str = "double-pendulum-dynamics"
    wandb_entity: str = ""            # "" -> your default wandb entity
    logdir: str = "runs"
    run_name: str = ""                # "" -> <modality>_<timestamp>
    log_interval: int = 1             # epochs
    eval_interval: int = 5
    save_interval: int = 10
    num_workers: int = 4
    # State modality only: load state+actions into RAM once so __getitem__
    # is pure numpy slicing (no per-window HDF5 gzip decompression). ~2.5 GB,
    # fits Colab RAM; makes the tiny state model GPU-bound. Use with
    # num_workers=0 (there is no I/O left to parallelise). Ignored for vision
    # (frames are far too large to preload).
    preload: bool = False
    device: str = "auto"

    def resolved_batch_size(self):
        """Return the effective batch size, substituting a modality default when unset.

        `batch_size=0` means "pick for me". Vision defaults smaller (64 vs 512)
        because each sample carries two 64x64x3 frames per timestep across the whole
        history window, so its memory/compute footprint dwarfs the state model's
        flat 203-dim vector; a smaller batch keeps GPU memory in check.
        """
        return self.batch_size or (512 if self.modality == "state" else 64)


## 4. State layout, target, and helpers

The env writes a 212-dim state per step (`env._get_info`):

```
[ basef_pos(3) | basef_R(6) | R_j0(6) | R_j1(6) | w0(3) | w1(3) |
  tau(6) | contacts(160) | filled(16) | gravity(3) ]
```

We predict the 18-dim **predictable** slice `[R_j0, R_j1, w0, w1]` (indices 9:27)
as an additive Δs. The state-model input is everything past the base (203 dims),
with the torque slot overwritten by the torque that *drives* the transition
(`actions[t+1]`, since actions are recorded post-step).

In [ ]:
PRED = slice(9, 27)          # 18-dim predictable state
TAU  = slice(27, 33)         # torque slot in the 212-vec
POST_BASE = slice(9, 212)    # drop base_pos + base_R -> 203 dims

STATE_INPUT_DIM = 203
PRED_DIM = 18
TORQUE_DIM = 6
ROT_DIM = 12                 # first 12 of PRED are the two 6D rotations
IMG = 64


def predictable(state):
    """Slice the 18-dim *predictable* sub-state ``[R_j0(6), R_j1(6), w0(3), w1(3)]``.

    NeRD predicts a residual state-difference Δs and reconstructs the next state as
    ``s_{t+1} = s_t + Δs`` rather than regressing the absolute next state, which is
    an easier learning target (the network only models *change*). We therefore only
    need to predict the parts of the state that actually evolve under the dynamics:
    the two joint orientations (as 6D rotations) and their angular velocities. Base
    position/orientation are omitted because the base is welded to the world, and
    contacts/torque/gravity are *inputs* the model conditions on, never outputs.

    Rotations use the continuous 6D representation (first two matrix columns) from
    Zhou et al. 2019, "On the Continuity of Rotation Representations in Neural
    Networks" (arXiv:1812.07035): unlike quaternions/Euler angles it has no
    discontinuities, so it is a far better regression target.

    Args:
        state: array ``[.., 212]`` following env._get_info's layout.
    Returns:
        array ``[.., 18]`` view/slice of the predictable components.
    """
    return state[..., PRED]


def build_state_input(cur, drive_tau):
    """Assemble one window of the state-model input token stream.

    Mirrors NeRD's per-step input token — ``{robot state, contact descriptors,
    joint torque, gravity}`` (Xu et al. 2025, arXiv:2508.15755) — for our system.
    We take the full 212-dim state minus the base pose (``state[9:212]`` → 203 dims,
    since base position/orientation are constant for a welded base) and overwrite
    its torque slot with the torque that actually *drives* the predicted transition.

    Why overwrite the torque: actions are recorded post-step, so ``state[t]`` carries
    the torque that produced ``state[t]`` (the previous transition), whereas the
    transition ``s_t -> s_{t+1}`` we are predicting is driven by ``actions[t+1]``.
    Feeding the correct driving torque is what makes this a causal one-step dynamics
    model rather than an off-by-one one.

    Args:
        cur:       ``[T, 212]`` "from" states of the T transitions in the window.
        drive_tau: ``[T, 6]`` torque driving each transition (i.e. ``actions[t+1]``).
    Returns:
        ``[T, 203]`` input array (contacts/gravity preserved, torque replaced).
    """
    x = cur[:, POST_BASE].copy()
    x[:, TAU.start - 9 : TAU.stop - 9] = drive_tau  # tau at [18:24] post-base
    return x


## 5. RunningMeanStd (matches DynamicsModel's `.normalize(x, un_norm)` API)

In [ ]:
class RunningMeanStd:
    """Fixed affine (de)normaliser: ``(x - mean) / std`` and its inverse.

    Standardising inputs and regression targets to zero-mean/unit-variance is a
    long-standing trick for conditioning neural-net training: it keeps gradients
    well-scaled and stops large-magnitude dimensions from dominating the loss. Here
    it matters because the state vector is deeply heterogeneous — 6D rotation
    entries live in [-1, 1] while angular velocities range over ±4π rad/s — so
    without normalisation the loss would be swamped by the velocity terms. NeRD
    (Xu et al. 2025) normalises its inputs and targets with exactly this kind of
    running-statistics module.

    The class exposes ``.normalize(x, un_norm)`` because DynamicsModel calls that
    signature internally (normalise inputs on the way in, de-normalise Δs on the way
    out). Stats are fitted once up front (see Trainer.compute_dataset_statistics)
    and then frozen — they are constants, not updated during training.
    """

    def __init__(self, mean, std):
        """Store the fixed mean/std, flooring std to avoid divide-by-zero on
        constant dimensions (e.g. a contact channel that is always empty)."""
        self.mean = mean.float()
        self.std = std.float().clamp_min(1e-6)

    def normalize(self, x, un_norm=False):
        """Standardise ``x`` (``un_norm=False``) or invert it back to raw units.

        Stats are moved onto ``x``'s device/dtype so the same object works on CPU
        and GPU. ``un_norm=True`` (``x*std + mean``) is used to turn the model's
        normalised Δs prediction back into physical state-delta units.
        """
        mean = self.mean.to(x.device, x.dtype)
        std = self.std.to(x.device, x.dtype)
        return x * std + mean if un_norm else (x - mean) / std

    def to_dict(self):
        """Serialise mean/std to plain Python lists for JSON-safe checkpointing."""
        return {"mean": self.mean.cpu().numpy().tolist(),
                "std": self.std.cpu().numpy().tolist()}

    @classmethod
    def from_dict(cls, d):
        """Rebuild a RunningMeanStd from a ``to_dict()`` payload (checkpoint load)."""
        return cls(torch.tensor(d["mean"]), torch.tensor(d["std"]))


class _Welford:
    """Streaming mean/variance accumulator over rows of a fixed-width vector.

    Implements Welford's online algorithm (B. P. Welford, 1962; popularised in
    Knuth, TAOCP Vol. 2, §4.2.2). Computing variance the naive way as
    ``E[x^2] - E[x]^2`` over millions of transitions accumulates catastrophic
    floating-point cancellation; Welford's single-pass update is numerically stable
    and lets us fit normalisation stats without holding the whole dataset in memory.
    """

    def __init__(self, dim):
        """Initialise running count, mean, and sum-of-squared-deviations (M2)."""
        self.n = 0
        self.mean = np.zeros(dim, dtype=np.float64)
        self.m2 = np.zeros(dim, dtype=np.float64)

    def update(self, batch):
        """Fold each row of ``batch`` into the running statistics one at a time."""
        for x in batch:
            self.n += 1
            d = x - self.mean
            self.mean += d / self.n
            self.m2 += d * (x - self.mean)

    def result(self):
        """Finalise into ``(mean, std)`` torch tensors (sample variance, ddof=1)."""
        var = self.m2 / max(self.n - 1, 1)
        return torch.tensor(self.mean), torch.tensor(np.sqrt(var))


## 6. TrajectoryDataset + window index

`build_index` scans only the cheap `obs` dataset to find each trajectory's valid
length (generate_data zero-pads after early termination) and enumerates every
length-`L` window. Trajectories are split train/val by a seeded permutation of
their global ids, so the split depends only on `seed` and is identical across
modalities.

In [ ]:
def valid_len(obs):
    """Count the leading non-padding rows of a trajectory's observation array.

    generate_data.py pre-allocates each trajectory to N_STEPS rows and zero-pads
    any steps after an early termination (a simulation blow-up). We must not train
    on those all-zero padding rows. A *valid* step always contains a unit quaternion
    in ``obs[:, 0:4]`` (L1 norm ≥ 1), whereas padding is exactly zero, so the first
    row that fails that test marks the end of real data.

    Args:
        obs: ``[N_STEPS, 14]`` observation array for one trajectory.
    Returns:
        int number of leading valid rows (== N_STEPS if the trajectory never
        terminated early).
    """
    good = np.abs(obs[:, 0:4]).sum(axis=-1) > 0.5
    return int(np.argmin(good)) if not good.all() else len(good)


def build_index(shard_paths, history, val_frac, seed):
    """Enumerate every training window and deterministically split train/val.

    A "window" is ``(shard_idx, traj_id, start)`` covering the ``history``
    consecutive transitions beginning at ``start`` within one trajectory — the
    causal transformer's context. Building an explicit index (rather than reading
    files to discover windows) lets the DataLoader random-access any window cheaply.

    The split is by *trajectory*, assigned via a seeded permutation of global
    trajectory ids, so (a) train and val never share timesteps from the same
    rollout (no leakage) and (b) the split depends only on ``seed`` — identical
    across the state and vision modalities, which is what makes their comparison
    fair. Only the small ``obs`` dataset is read here to find valid lengths; the
    heavy state/rgb datasets are never touched, so indexing is fast.

    Args:
        shard_paths: sorted list of shard_*.h5 paths.
        history:     window length (transitions per sample).
        val_frac:    fraction of trajectories held out for validation.
        seed:        RNG seed controlling the split.
    Returns:
        ``(train_windows, val_windows)`` as int32 arrays of shape ``[N, 3]``.
    """
    rng = np.random.default_rng(seed)
    per_shard = []
    n_total = 0
    for path in shard_paths:
        with h5py.File(path, "r") as f:
            keys = sorted(k for k in f if k.startswith("traj_"))
            lens = [valid_len(f[k]["obs"][:]) for k in keys]
        per_shard.append(lens)
        n_total += len(lens)

    perm = rng.permutation(n_total)
    val_ids = set(perm[: int(round(val_frac * n_total))].tolist())

    def to_int32(windows):
        """Pack a list of (shard, traj, start) tuples into an int32 [N,3] array."""
        return np.asarray(windows, dtype=np.int32).reshape(-1, 3)

    train, val = [], []
    gid = 0
    for shard_idx, lens in enumerate(per_shard):
        for traj_id, L in enumerate(lens):
            bucket = val if gid in val_ids else train
            for s in range(0, L - 1 - history + 1):  # need start+history <= L-1
                bucket.append((shard_idx, traj_id, s))
            gid += 1
    return to_int32(train), to_int32(val)


class TrajectoryDataset(Dataset):
    """One length-`history` window per item. HDF5 handles opened lazily per worker."""

    def __init__(self, shard_paths, windows, history, modality, preload=False):
        """Store the window index and file list; defer opening HDF5 files.

        Files are intentionally *not* opened here. With multi-worker DataLoaders the
        dataset object is forked into each worker process, and an h5py handle opened
        before the fork would be shared and corrupt reads; opening lazily inside the
        worker (see ``_file``) gives every worker its own clean handle.

        ``preload`` (state modality only): read every referenced trajectory's
        state+actions into RAM once, so __getitem__ becomes pure numpy slicing with
        no per-window HDF5 gzip decompression — the fix for a GPU that is starving
        on CPU-bound decompression. Pair with num_workers=0 (no I/O to parallelise,
        and it avoids forking the multi-GB store into every worker).
        """
        self.shard_paths = shard_paths
        self.windows = windows
        self.history = history
        self.modality = modality
        self._files = {}
        self._store = None
        if preload and modality == "state":
            self._store = {}
            files = {}
            try:
                for w in windows:
                    key = (int(w[0]), int(w[1]))
                    if key in self._store:
                        continue
                    si = key[0]
                    fh = files.get(si)
                    if fh is None:
                        fh = h5py.File(shard_paths[si], "r")
                        files[si] = fh
                    g = fh[f"traj_{key[1]:04d}"]
                    # keep native float64 so the Δs target is computed exactly as
                    # in the HDF5 path (which casts to float32 only at the end)
                    self._store[key] = (g["state"][:], g["actions"][:])
            finally:
                for fh in files.values():
                    fh.close()
            print(f"preloaded {len(self._store)} trajectories into RAM")

    def _file(self, shard_idx):
        """Return this worker's cached h5py handle for a shard, opening it on first use."""
        f = self._files.get(shard_idx)
        if f is None:
            f = h5py.File(self.shard_paths[shard_idx], "r")
            self._files[shard_idx] = f
        return f

    def __del__(self):
        """Close shard handles deterministically. In a long-lived Colab kernel the
        run cell gets re-executed many times, each building fresh datasets; without
        this, stale handles linger until GC and open fds accumulate."""
        for f in self._files.values():
            try:
                f.close()
            except Exception:
                pass

    def __len__(self):
        """Number of windows (== number of training samples per epoch pass)."""
        return len(self.windows)

    def __getitem__(self, i):
        """Load one window and build its (input, target) for the chosen modality.

        Reads the ``history+1`` states spanning the window's transitions, forms the
        additive Δs target ``predictable(nxt) - predictable(cur)`` (dense: one target
        per position, matching how the causal GPT emits a prediction at every
        timestep — the same next-step supervision that trains GPT-style models,
        Radford et al. 2019), and packages the modality-specific inputs:

          * state  — the 203-dim NeRD-style token (see build_state_input);
          * vision — the two camera views scaled to [-1, 1] plus the driving torque
            (the torque cannot be inferred from pixels, so it is fed alongside).

        The transition ``s_t -> s_{t+1}`` is driven by ``actions[t+1]`` (actions are
        recorded post-step), which is why ``drive_tau`` is the shifted action slice.
        """
        shard_idx, traj_id, s = (int(v) for v in self.windows[i])
        T = self.history

        if self._store is not None:                       # preloaded (state only)
            fs, fa = self._store[(shard_idx, traj_id)]
            state = fs[s : s + T + 1]                      # [T+1, 212]
            actions = fa[s : s + T + 1]                    # [T+1, 6]
        else:
            g = self._file(shard_idx)[f"traj_{traj_id:04d}"]
            state = g["state"][s : s + T + 1]
            actions = g["actions"][s : s + T + 1]
        cur, nxt = state[:-1], state[1:]
        drive_tau = actions[1:]                # actions[s+1..s+T] drives each step

        target = (predictable(nxt) - predictable(cur)).astype(np.float32)
        out = {"target": torch.from_numpy(target)}

        if self.modality == "state":
            x = build_state_input(cur, drive_tau).astype(np.float32)
            out[STATE_KEY] = torch.from_numpy(x)
        else:
            g = self._file(shard_idx)[f"traj_{traj_id:04d}"]
            cam0 = g["rgb_cam0"][s : s + T]
            cam1 = g["rgb_cam1"][s : s + T]
            frames = np.stack([cam0, cam1], 1).astype(np.float32) / 127.5 - 1.0
            frames = np.transpose(frames, (0, 1, 4, 2, 3))  # [T,2,3,64,64]
            out[FRAMES_KEY] = torch.from_numpy(np.ascontiguousarray(frames))
            out[TORQUE_KEY] = torch.from_numpy(drive_tau.astype(np.float32))
        return out


## 7. Trainer  — mirrors `VanillaTrainer` / `SequenceModelTrainer`

In [ ]:
def pick_device(requested):
    """Resolve the compute device, auto-selecting the best available accelerator.

    ``requested`` is honoured verbatim unless it is ``"auto"``, in which case we
    prefer CUDA (Colab's T4/A100), then Apple MPS (local Mac dev), then CPU. The
    vision encoder's CNN is impractically slow on CPU/MPS, so real vision runs
    belong on CUDA — this just keeps the notebook runnable everywhere for
    wiring/smoke checks.
    """
    if requested != "auto":
        return requested
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class Trainer:
    """NeRD-style trainer for the double-pendulum dynamics model.

    __init__ builds datasets, normalisation stats, model, optimizer, LR schedule,
    and wandb logging (the same responsibilities as VanillaTrainer.__init__).
    """

    def __init__(self, cfg: Config):
        """Wire up a full training run from a Config: seeds, data, stats, model, logging.

        Seeds are set first for reproducibility, then the run directory + loggers are
        created (wandb if enabled, always a local metrics.jsonl mirror), then the
        datasets, normalisation statistics, and model/optimizer are built in that
        order (stats must exist before the model so its normalisers can be attached).
        """
        self.cfg = cfg
        self.device = pick_device(cfg.device)
        torch.manual_seed(cfg.seed)
        np.random.seed(cfg.seed)

        self.shard_paths = sorted(glob.glob(os.path.join(cfg.data_dir, "shard_*.h5")))
        if not self.shard_paths:
            raise FileNotFoundError(f"No shard_*.h5 in {cfg.data_dir}/")

        run = cfg.run_name or f"{cfg.modality}_{time.strftime('%Y%m%d_%H%M%S')}"
        self.log_dir = os.path.join(cfg.logdir, run)
        os.makedirs(self.log_dir, exist_ok=True)
        self.metrics_f = open(os.path.join(self.log_dir, "metrics.jsonl"), "a")

        self.use_wandb = cfg.use_wandb and wandb is not None
        if cfg.use_wandb and wandb is None:
            print("wandb not installed - logging to metrics.jsonl only")
        if self.use_wandb:
            wandb.init(project=cfg.wandb_project, entity=(cfg.wandb_entity or None),
                       name=run, dir=self.log_dir, config=asdict(cfg))

        self.get_datasets()
        if cfg.compute_dataset_statistics:
            self.compute_dataset_statistics()
        self.build_model()

        self.best_val = float("inf")
        print(f"device={self.device} | modality={cfg.modality} | out={self.log_dir}")

    # ------------------------------------------------------------------ #
    def get_datasets(self):
        """Build the train/val window index, datasets, and DataLoaders.

        Named ``get_datasets`` to mirror NeRD's ``SequenceModelTrainer.get_datasets``,
        which likewise constructs a sequence-sampling ``TrajectoryDataset``. The
        train loader shuffles and drops the last partial batch (steady batch sizes
        keep BatchNorm/optimizer statistics well-behaved); the val loader does not.

        Validation is scored on a fixed, seeded random subset of the val windows
        sized to exactly ``num_val_batches`` batches. Drawing batches from the front
        of an unshuffled loader would score the same earliest trajectories every
        epoch; a fixed random subset is representative of the whole split and,
        being frozen, keeps val curves comparable across epochs.
        """
        cfg = self.cfg
        self.train_windows, self.val_windows = build_index(
            self.shard_paths, cfg.sample_sequence_length, cfg.val_frac, cfg.seed
        )
        print(f"windows: train={len(self.train_windows):,} val={len(self.val_windows):,}")
        if len(self.train_windows) == 0:
            raise RuntimeError("No training windows — sample_sequence_length too long?")

        bs = cfg.resolved_batch_size()
        max_eval = cfg.num_val_batches * bs
        if len(self.val_windows) > max_eval:
            rng = np.random.default_rng(cfg.seed)
            self.val_windows = self.val_windows[
                rng.choice(len(self.val_windows), max_eval, replace=False)]
            print(f"validation subset: {len(self.val_windows):,} windows "
                  f"({cfg.num_val_batches} batches)")

        def make_dataset(windows):
            """Wrap a window index in a TrajectoryDataset for the current modality."""
            return TrajectoryDataset(self.shard_paths, windows,
                                     cfg.sample_sequence_length, cfg.modality,
                                     preload=cfg.preload)

        self.train_dataset = make_dataset(self.train_windows)
        self.valid_dataset = make_dataset(self.val_windows)

        common = dict(num_workers=cfg.num_workers, pin_memory=(self.device == "cuda"))
        self.train_loader = DataLoader(self.train_dataset, bs, shuffle=True, drop_last=True, **common)
        self.valid_loader = DataLoader(self.valid_dataset, bs, shuffle=False, drop_last=False, **common)

    # ------------------------------------------------------------------ #
    def compute_dataset_statistics(self):
        """Fit the input/output normalisation statistics before training.

        Estimates per-dimension mean/std for (a) the model input and (b) the Δs
        target, from a random sample of ``stat_trajs`` training trajectories via
        Welford's algorithm. These freeze the RunningMeanStd normalisers used
        throughout training (see RunningMeanStd for why normalisation matters, and
        NeRD's own ``compute_dataset_statistics``). Sampling a subset — rather than
        all ~3M transitions — is plenty for a stable mean/std and keeps startup fast.

        Only the small ``state``/``actions`` datasets are read (never the rgb frames),
        so this is cheap even for the vision model; vision frames are normalised by a
        fixed [-1, 1] rescale in the dataset instead of by fitted statistics.
        """
        cfg = self.cfg
        rng = np.random.default_rng(cfg.seed)
        pairs = np.unique(self.train_windows[:, :2], axis=0)
        if len(pairs) > cfg.stat_trajs:
            pairs = pairs[rng.choice(len(pairs), cfg.stat_trajs, replace=False)]

        in_dim = STATE_INPUT_DIM if cfg.modality == "state" else TORQUE_DIM
        inp, out = _Welford(in_dim), _Welford(PRED_DIM)
        files = {}
        try:
            for shard_idx, traj_id in tqdm(pairs, desc="stats"):
                f = files.setdefault(int(shard_idx), h5py.File(self.shard_paths[int(shard_idx)], "r"))
                g = f[f"traj_{int(traj_id):04d}"]
                L = valid_len(g["obs"][:])
                if L < 2:
                    continue
                state, actions = g["state"][:L], g["actions"][:L]
                cur, nxt, drive_tau = state[:-1], state[1:], actions[1:]
                out.update(predictable(nxt) - predictable(cur))
                inp.update(build_state_input(cur, drive_tau) if cfg.modality == "state" else drive_tau)
        finally:
            for f in files.values():
                f.close()

        in_mean, in_std = inp.result()
        out_mean, out_std = out.result()
        key = STATE_KEY if cfg.modality == "state" else TORQUE_KEY
        self.input_rms = {key: RunningMeanStd(in_mean, in_std)}
        self.output_rms = RunningMeanStd(out_mean, out_std)
        self._out_std = self.output_rms.std.to(self.device)

    # ------------------------------------------------------------------ #
    def build_model(self):
        """Construct the DynamicsModel (encoder + causal GPT + Δs head) and optimizer.

        The encoder is chosen by modality — an identity MLP over the 203-dim state
        token, or the two-view CNN VisionEncoder — feeding a shared GPT-2-style
        causal transformer core (the NeRD "ModelMixedInput" backbone). If stats were
        fitted, they are attached so the model (de)normalises internally.

        Optimiser is AdamW (Loshchilov & Hutter 2019, "Decoupled Weight Decay
        Regularization", arXiv:1711.05101). Following common transformer practice
        (e.g. nanoGPT, which this GPT core is adapted from), weight decay is applied
        only to ≥2-D tensors (matmul/embedding weights) and *not* to biases or
        LayerNorm gains — decaying those tends to hurt. ``betas`` default to
        (0.9, 0.95), the GPT-style setting for the second moment.
        """
        cfg = self.cfg
        gpt = GPTConfig(block_size=max(cfg.sample_sequence_length, cfg.block_size),
                        n_layer=cfg.n_layer, n_head=cfg.n_head, n_embd=cfg.n_embd,
                        dropout=cfg.dropout, bias=True)
        if cfg.modality == "state":
            mcfg = DynamicsModelConfig(
                encoder=MLPEncoderConfig(layer_sizes=[]),  # identity; GPT.wte projects
                dynamics=gpt, head=MLPEncoderConfig(layer_sizes=list(cfg.head_hidden), activation="relu"), history_length=cfg.sample_sequence_length,
                normalize_input=cfg.normalize_input, normalize_output=cfg.normalize_output)
            self.model = DynamicsModel(mcfg, output_dim=PRED_DIM,
                                       input_size=STATE_INPUT_DIM, device=self.device)
        else:
            mcfg = DynamicsModelConfig(
                encoder=VisionEncoderConfig(image_size=IMG),
                dynamics=gpt, head=MLPEncoderConfig(layer_sizes=list(cfg.head_hidden), activation="relu"), history_length=cfg.sample_sequence_length,
                normalize_input=cfg.normalize_input, normalize_output=cfg.normalize_output)
            self.model = DynamicsModel(mcfg, output_dim=PRED_DIM,
                                       torque_dim=TORQUE_DIM, device=self.device)

        if hasattr(self, "output_rms"):
            self.model.set_input_rms(self.input_rms)
            self.model.set_output_rms(self.output_rms)
        else:
            self._out_std = torch.ones(PRED_DIM, device=self.device)

        decay = [p for p in self.model.parameters() if p.requires_grad and p.dim() >= 2]
        nodecay = [p for p in self.model.parameters() if p.requires_grad and p.dim() < 2]
        self.optimizer = torch.optim.AdamW(
            [{"params": decay, "weight_decay": cfg.weight_decay},
             {"params": nodecay, "weight_decay": 0.0}],
            lr=cfg.lr_start, betas=cfg.betas)

        # fp16 autocast + loss scaling, active only for training steps on CUDA.
        # GradScaler(enabled=False) degrades to a transparent no-op, so the same
        # code path runs unchanged on MPS/CPU.
        self.amp_enabled = cfg.use_amp and self.device == "cuda"
        self.scaler = torch.amp.GradScaler("cuda", enabled=self.amp_enabled)
        print(f"model parameters: {sum(p.numel() for p in self.model.parameters()):,}"
              f"{' | amp: fp16' if self.amp_enabled else ''}")

    # ------------------------------------------------------------------ #
    def get_scheduled_learning_rate(self, epoch):
        """Return the learning rate for ``epoch`` under the configured schedule.

        A learning-rate schedule — a high LR early to make fast progress, decayed
        later to settle into a sharp minimum — is standard for stable neural-net
        training and is used by NeRD as well. We offer three:

          * ``constant`` — no decay (a baseline / debugging aid);
          * ``linear``   — straight-line interpolation from ``lr_start`` to ``lr_end``;
          * ``cosine``   — cosine annealing from ``lr_start`` down to ``lr_end``, the
            default. Cosine decay comes from Loshchilov & Hutter 2017, "SGDR:
            Stochastic Gradient Descent with Warm Restarts" (arXiv:1608.03983); its
            smooth, slow-at-the-ends shape reliably outperforms step decay and is the
            de-facto choice for transformer training.

        Args:
            epoch: current epoch index in ``[0, num_epochs)``.
        Returns:
            float learning rate to set on the optimizer's param groups.
        """
        cfg = self.cfg
        frac = epoch / max(cfg.num_epochs - 1, 1)
        if cfg.lr_schedule == "constant":
            return cfg.lr_start
        if cfg.lr_schedule == "linear":
            return cfg.lr_start + (cfg.lr_end - cfg.lr_start) * frac
        return cfg.lr_end + 0.5 * (cfg.lr_start - cfg.lr_end) * (1 + math.cos(math.pi * frac))

    def preprocess_data_batch(self, data):
        """Move a collated batch onto the device and split it into (inputs, target).

        Named after NeRD's ``preprocess_data_batch``. The input dict is keyed to
        match what DynamicsModel expects for each modality (a single state tensor,
        or frames + torque). The target is the additive Δs the model regresses.
        """
        if self.cfg.modality == "state":
            inp = {STATE_KEY: data[STATE_KEY].to(self.device)}
        else:
            inp = {FRAMES_KEY: data[FRAMES_KEY].to(self.device),
                   TORQUE_KEY: data[TORQUE_KEY].to(self.device)}
        return inp, data["target"].to(self.device)

    def compute_loss(self, data):
        """Forward pass + dense, variance-weighted MSE on the Δs prediction.

        The model emits a raw Δs at every position of the window (dense supervision;
        the causal mask means each position only attends to its past, so all T
        predictions are valid one-step targets — the same principle as next-token
        prediction in GPT). We divide the residual by the per-dimension output std
        before squaring — "variance weighting", as in NeRD's ``compute_loss``. This
        is equivalent to computing MSE in normalised space: it prevents the
        large-magnitude angular-velocity components from dominating the small-
        magnitude rotation components, so every state dimension is learned equally.

        The itemized dict reports *raw* (physical-unit) MSE overall and split into
        rotation vs. angular-velocity error — a useful diagnostic for spotting when
        one part of the state is being learned far better than the other.

        When ``cfg.inject_noise`` is on, training-mode forward passes perturb the
        normalised inputs with small Gaussian noise (NeRD's robustness trick for
        autoregressive rollout); eval-mode passes are always clean.

        Returns:
            ``(loss, itemized)`` where ``loss`` is the scalar training objective.
        """
        inp, target = self.preprocess_data_batch(data)
        # inject_noise only while training (one_epoch toggles model.train/eval):
        # validation and inference always see clean inputs.
        pred = self.model(inp, inject_noise=(self.cfg.inject_noise and self.model.training))  # raw Δs [B,T,18]
        resid = pred - target
        loss = (resid / self._out_std).pow(2).mean()   # normalised (variance-weighted)
        with torch.no_grad():
            itemized = {"raw_mse": resid.pow(2).mean().item(),
                        "rot_mse": resid[..., :ROT_DIM].pow(2).mean().item(),
                        "angvel_mse": resid[..., ROT_DIM:].pow(2).mean().item()}
        return loss, itemized

    # ------------------------------------------------------------------ #
    def one_epoch(self, train, num_batches, loader_iter):
        """Run one epoch of ``num_batches`` steps in either train or eval mode.

        Mirrors NeRD's ``one_epoch(train=...)`` — a single method used for both
        training and validation, toggled by the flag. An "epoch" here is a fixed
        number of batches drawn from an *infinite* shuffled iterator (the window set
        is far too large to sweep fully), so ``num_batches`` sets the work per epoch
        rather than the dataset defining it.

        In training mode it does the usual zero-grad → backward → step, with gradient
        clipping in between. Clipping the global grad norm (Pascanu et al. 2013, "On
        the difficulty of training recurrent neural networks", arXiv:1211.5063)
        guards against the occasional exploding gradient — especially relevant for
        contact dynamics, whose near-discontinuous impacts produce sharp, stiff loss
        landscapes (Bianchini et al. 2021, "Fundamental Challenges in Deep Learning
        for Stiff Contact Dynamics", arXiv:2103.15406) that can spike gradients.

        Returns:
            ``(avg_loss, avg_itemized, avg_grad_norm)`` over the epoch.
        """
        self.model.train(train)
        tot = 0.0
        item = {"raw_mse": 0.0, "rot_mse": 0.0, "angvel_mse": 0.0}
        grad_acc, n = 0.0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for _ in tqdm(range(num_batches), leave=False, desc="train" if train else "valid"):
                data = next(loader_iter)
                # fp16 autocast for training steps only; validation stays fp32.
                with torch.autocast("cuda", dtype=torch.float16,
                                    enabled=self.amp_enabled and train):
                    loss, it = self.compute_loss(data)
                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                    self.scaler.scale(loss).backward()
                    if self.cfg.truncate_grad:
                        # Gradients must be unscaled before clipping, otherwise the
                        # norm is measured on scaled values and the clip is wrong.
                        self.scaler.unscale_(self.optimizer)
                        gnorm = nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_norm)
                        grad_acc += float(gnorm)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                tot += loss.item()
                for k in item:
                    item[k] += it[k]
                n += 1
        avg = tot / max(n, 1)
        item = {k: v / max(n, 1) for k, v in item.items()}
        return avg, item, grad_acc / max(n, 1)

    # ------------------------------------------------------------------ #
    def train(self):
        """Main training loop: epochs of (schedule LR → train → validate → log/save).

        Follows NeRD's ``VanillaTrainer.train`` structure. Each epoch: set the
        scheduled learning rate, run a training epoch, run a validation epoch, then
        log/checkpoint at the configured cadences, tracking the best validation loss.

        Epoch 0 deliberately skips training and only validates — this records the
        model's loss at initialisation as a baseline (also NeRD's behaviour), so the
        first logged training epoch can be compared against an untrained reference.
        """
        cfg = self.cfg

        def infinite(loader):
            """Yield batches forever by looping the DataLoader.

            The window set is enormous, so we treat "epoch" as a fixed batch budget
            rather than a full pass; this generator re-iterates the shuffled loader
            endlessly to supply that budget.
            """
            while True:
                yield from loader

        train_iter = infinite(self.train_loader)
        # Validation gets a *fresh finite* iterator every epoch (below), so each
        # pass sweeps the same fixed subset exactly once — never duplicates within
        # an epoch, never drifts out of alignment across epochs.
        val_batches = min(cfg.num_val_batches, len(self.valid_loader))

        for epoch in range(cfg.num_epochs):
            lr = self.get_scheduled_learning_rate(epoch)
            for pg in self.optimizer.param_groups:
                pg["lr"] = lr

            train_loss, train_item, grad = (float("nan"), {}, 0.0)
            if epoch > 0:  # NeRD skips training at epoch 0 (baseline validation)
                train_loss, train_item, grad = self.one_epoch(True, cfg.num_batches_per_epoch, train_iter)

            val_loss, val_item, _ = self.one_epoch(False, val_batches, iter(self.valid_loader))

            if epoch % cfg.log_interval == 0:
                rec = {"epoch": epoch, "lr": lr, "train_norm_mse": train_loss,
                       "valid_norm_mse": val_loss, "grad_norm": grad,
                       "train": train_item, "valid": val_item}
                self.metrics_f.write(json.dumps(rec) + "\n")
                self.metrics_f.flush()
                if self.use_wandb:
                    flat = {"loss/train": train_loss, "loss/valid": val_loss, "lr": lr,
                            "grad_norm": grad,
                            **{f"valid/{k}": v for k, v in val_item.items()},
                            **{f"train/{k}": v for k, v in train_item.items()}}
                    wandb.log(flat, step=epoch)
                print(f"epoch {epoch:>4} | lr {lr:.2e} | train {train_loss:.4f} | "
                      f"valid {val_loss:.4f} (raw {val_item['raw_mse']:.3f})")

            if val_loss < self.best_val:
                self.best_val = val_loss
                self.save_model("best_model")

            if (epoch + 1) % cfg.save_interval == 0:
                self.save_model(f"model_epoch{epoch + 1}")

        self.save_model("final_model")
        self.metrics_f.close()
        if self.use_wandb:
            wandb.finish()
        print(f"done. best valid_norm_mse={self.best_val:.4f} | {self.log_dir}")

    # ------------------------------------------------------------------ #
    def save_model(self, name):
        """Checkpoint everything needed to resume or run inference.

        Saves the model + optimizer state, the fitted normalisation stats, the full
        config, and the best validation score to ``runs/<run>/<name>.pt``. The
        normalisers are stored alongside the weights because a trained model is
        unusable without them — inference must standardise inputs and de-standardise
        the Δs output with the *same* statistics used at train time.
        """
        path = os.path.join(self.log_dir, f"{name}.pt")
        torch.save({
            "model": self.model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "input_rms": {k: v.to_dict() for k, v in getattr(self, "input_rms", {}).items()},
            "output_rms": self.output_rms.to_dict() if hasattr(self, "output_rms") else None,
            "scaler": self.scaler.state_dict(),
            "config": asdict(self.cfg),
            "best_val": self.best_val,
        }, path)

    def load_model(self, path):
        """Restore model + optimizer + normalisers + best-score from a checkpoint.

        Assumes the current Trainer was built with a matching config/modality (so the
        module shapes line up). ``map_location`` retargets tensors to this Trainer's
        device, letting a GPU-trained checkpoint load on CPU and vice versa.

        The normalisation statistics are restored from the checkpoint — not refit
        from the current dataset — because the weights are only meaningful with the
        exact stats they were trained under; refitting would silently mis-scale the
        model if the data had changed between runs.
        """
        ck = torch.load(path, map_location=self.device)
        self.model.load_state_dict(ck["model"])
        self.optimizer.load_state_dict(ck["optimizer"])
        if ck.get("input_rms"):
            self.input_rms = {k: RunningMeanStd.from_dict(v)
                              for k, v in ck["input_rms"].items()}
            self.model.set_input_rms(self.input_rms)
        if ck.get("output_rms"):
            self.output_rms = RunningMeanStd.from_dict(ck["output_rms"])
            self.model.set_output_rms(self.output_rms)
            self._out_std = self.output_rms.std.to(self.device)
        if ck.get("scaler"):
            self.scaler.load_state_dict(ck["scaler"])
        self.best_val = ck.get("best_val", float("inf"))
        print(f"loaded {path} (normalisers restored from ckpt)")


## 8. Run

Mirrors NeRD's `train/train.py` bottom: build the config, construct the trainer,
call `train()`. Flip `modality` to `"vision"` for the vision model (train that one
on a GPU — the CNN encoder is slow on CPU/MPS).

In [ ]:
cfg = Config(
    modality="state",          # "state" | "vision"
    data_dir="data",           # point at your shard folder (Drive path on Colab)
    num_epochs=100,
    num_batches_per_epoch=500,
    lr_schedule="cosine",
)
# On Colab you might override: cfg.data_dir = "/content/drive/MyDrive/.../data"

trainer = Trainer(cfg)
trainer.train()


## 9. Weights & Biases

Log in once per Colab session before the run cell (grab your key from
<https://wandb.ai/authorize>):

```python
import wandb; wandb.login()
```

Set `cfg.wandb_project` / `cfg.wandb_entity` as needed; each run streams to wandb
under the same name as its `runs/<name>/` folder. Metrics are always mirrored to
`metrics.jsonl` locally too. Checkpoints (`best_model.pt`, `final_model.pt`,
`model_epochN.pt`) land under `runs/<name>/` — on Colab, set `cfg.logdir` to a
Drive path (or copy `runs/` to Drive afterward) so they survive VM recycling.

**Quick wiring check** (skip wandb) before a long run:

```python
smoke = Config(modality="state", num_epochs=2, num_batches_per_epoch=10,
               num_val_batches=5, stat_trajs=50, num_workers=0, use_wandb=False)
Trainer(smoke).train()
```